# 03 - Content Embeddings

The production code in `src/features/content_embeddings.py` uses
`sentence-transformers` (SBERT, `all-MiniLM-L6-v2`) to turn each track's
text (name + artist + album) into a dense semantic vector. That's the right
choice for a real deployment - it captures semantic similarity even when
tracks don't share exact words.

**In this notebook** we use scikit-learn's `TfidfVectorizer` instead, purely
so the notebook runs end-to-end without needing to download model weights
from Hugging Face. TF-IDF and SBERT are interchangeable at the interface
level here - both produce one fixed-length vector per track, and everything
downstream (similarity search, cold-start routing) works the same either
way. Swap in `SpotifyTrackVectorizer` from `src/features/content_embeddings.py`
for the semantically richer SBERT vectors in an environment with internet
access.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tracks = pd.read_csv('../data/processed/tracks_clean.csv')
print(tracks.shape)
tracks[['track_name', 'artists', 'album_name', 'track_genre']].head()

(16804, 21)


,track_name,artists,album_name,track_genre
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,Unholy (feat. Kim Petras),dance
1,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",hip-hop
2,La Bachata,Manuel Turizo,La Bachata,latin
3,I'm Good (Blue),David Guetta;Bebe Rexha,I'm Good (Blue),dance
4,Tití Me Preguntó,Bad Bunny,Un Verano Sin Ti,latin


In [2]:
tracks['content_text'] = (
    tracks['track_name'].astype(str) + ' ' +
    tracks['artists'].astype(str) + ' ' +
    tracks['album_name'].astype(str) + ' ' +
    tracks['track_genre'].astype(str)
)
tracks['content_text'].head()

0    Unholy (feat. Kim Petras) Sam Smith;Kim Petras...
1    Quevedo: Bzrp Music Sessions, Vol. 52 Bizarrap...
2            La Bachata Manuel Turizo La Bachata latin
3    I'm Good (Blue) David Guetta;Bebe Rexha I'm Go...
4    Tití Me Preguntó Bad Bunny Un Verano Sin Ti latin
Name: content_text, dtype: str

In [3]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
content_matrix = vectorizer.fit_transform(tracks['content_text'])
print("Content matrix shape:", content_matrix.shape)

Content matrix shape: (16804, 5000)


## Sanity check: find similar tracks to a known song

In [4]:
def most_similar(track_name, top_n=5):
    matches = tracks.index[tracks['track_name'].str.lower() == track_name.lower()]
    if len(matches) == 0:
        print(f"'{track_name}' not found")
        return
    idx = matches[0]
    sims = cosine_similarity(content_matrix[idx], content_matrix).flatten()
    top_idx = sims.argsort()[::-1][1:top_n + 1]
    print(f"Tracks most similar to: {tracks.loc[idx, 'track_name']} - {tracks.loc[idx, 'artists']}\n")
    for i in top_idx:
        print(f"  {sims[i]:.3f}  {tracks.loc[i, 'track_name']} - {tracks.loc[i, 'artists']}")

most_similar(tracks.loc[0, 'track_name'])

Tracks most similar to: Unholy (feat. Kim Petras) - Sam Smith;Kim Petras

  0.690  Kids In America - Kim Wilde
  0.452  Notorious B.I.G. (feat. Lil' Kim & Puff Daddy) - 2005 Remaster - The Notorious B.I.G.;Lil' Kim;Diddy
  0.407  Fire On Fire - Sam Smith
  0.394  Promises (with Sam Smith) - Calvin Harris;Sam Smith;Jessie Reyez
  0.393  Sevmekten Kim Usanır - Müzeyyen Senar


## Save embeddings for the next notebook

We save a compact `.npz` (dense array + track_uri index) rather than the full sparse TF-IDF matrix, since downstream cosine-similarity code expects a dense array per track (matching the interface `SpotifyTrackVectorizer.generate_track_embeddings` returns).

In [5]:
from sklearn.decomposition import TruncatedSVD

# Reduce to a dense, fixed-size vector per track (mirrors the 384-dim
# dense output shape you'd get from all-MiniLM-L6-v2, just smaller here
# since TF-IDF has less to compress).
svd = TruncatedSVD(n_components=100, random_state=42)
dense_embeddings = svd.fit_transform(content_matrix)
print("Dense embeddings shape:", dense_embeddings.shape)
print("Explained variance captured:", svd.explained_variance_ratio_.sum().round(3))

tracks['track_uri'] = 'spotify:track:' + tracks['track_id'].astype(str)
np.savez('../data/processed/content_embeddings.npz',
         embeddings=dense_embeddings,
         track_uris=tracks['track_uri'].values)
print("Saved data/processed/content_embeddings.npz")

Dense embeddings shape: (16804, 100)
Explained variance captured: 0.25
Saved data/processed/content_embeddings.npz
